In [ ]:
import os
import sys
path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(path)
print(path)

from utility import*

import numpy as np
import random
from collections import Counter
import os

In [ ]:

def generate_traffic_uniform(Nrack, Hosts_p_rack, loadfrac0, totaltime, Gbps_rate, Nactive, workload, network, seedValue):
    print(f'Running Python function with arguments: {Nrack}, {Hosts_p_rack}, {loadfrac0}, {totaltime}, {Gbps_rate}, {Nactive}, {workload}, {network}, {seedValue}')

    filename = f'{network}/{workload}_{100 * loadfrac0:.2f}percLoad_{int(totaltime)}sec_{Nrack}N_{Hosts_p_rack}hpr_{Nrack * Hosts_p_rack}hosts_{Gbps_rate}Gbps_{Nactive:.2f}Nactive_seed={seedValue}.htsim'

    if os.path.exists(filename):
        print(f"File {filename} already exists. Skipping the process.")
        return
    
    np.random.seed(seedValue)
    random.seed(seedValue) 

    H_active = int(np.ceil(Nrack * Hosts_p_rack * Nactive))
    print(f'H_active = {H_active}')

    probabilities, srcdst = get_uniform_probabilities(H_active, Hosts_p_rack)
    flowmat1 = get_flow_mat(probabilities, srcdst, workload, Gbps_rate, loadfrac0, H_active, totaltime)

    # Write the flowmat1 to the file in the appropriate format
    write_to_htsim_file(flowmat1, filename)



def get_uniform_probabilities(H_active, Hosts_p_rack):

    Ncons = H_active * (H_active - Hosts_p_rack)  # number of possible connections
    srcdst = np.zeros((Ncons, 2), dtype=int)
    
    # Initialize the probability list
    probabilities = []
    cnt = 0
    for a in range(H_active):  # sources
        for b in range(H_active):  # destinations
            if a // Hosts_p_rack != b // Hosts_p_rack:
                
                # Store the source-destination pair
                srcdst[cnt] = [a, b]
                probabilities.append(1.0)
                cnt += 1
    
    return probabilities, srcdst



In [ ]:
network_ = "opera"

if network_ == "clos":
    Nrack_ = 72
    Hosts_p_rack_ = 9

elif network_ == "opera":
    Nrack_ = 108
    Hosts_p_rack_ = 6

workload_ = "HD"
time_ = 10.001
Gbps_rate_ = 40
Nactive_ = 1
seedValue_ = 1

load_set = [0.02, 0.04, 0.05, 0.06, 0.08, 0.10]
seed_set = [1,2,3, 4, 5]
for seedValue_ in seed_set:
    for load_ in load_set:
        generate_traffic_uniform(Nrack_, Hosts_p_rack_, load_, time_, Gbps_rate_, Nactive_, workload_, network_, seedValue_)
